In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
df = pd.read_csv(url, names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
from sklearn.model_selection import train_test_split
X=df.drop('Outcome',axis=1)
y=df['Outcome']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train = X_train.copy()
X_test = X_test.copy()

In [3]:
nancols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']
X_train[nancols] = X_train[nancols].replace(0,np.nan)
X_train.head()
X_test[nancols] = X_test[nancols].replace(0,np.nan)
X_test.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
668,6,98,58.0,33.0,190.0,34.0,0.430,43
324,2,112,75.0,32.0,NaN,35.7,0.148,21
624,2,108,64.0,NaN,NaN,30.8,0.158,21
690,8,107,80.0,NaN,NaN,24.6,0.856,34
473,7,136,90.0,NaN,NaN,29.9,0.210,50


In [4]:
#time to impute the NaNs with the median
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

In [5]:
#scaling values
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_std = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train.columns, index=X_train.index)
X_test_std = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test.columns, index=X_test.index)
X_train_std.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
60,-0.526397,-1.257201,-0.018995,-0.008140,-0.204516,-0.050247,-0.490735,-1.035940
618,1.588046,-0.326334,0.808174,-0.543652,-0.204516,-0.598590,2.415030,1.487101
346,-0.828460,0.571288,-2.169636,-1.138665,-0.622227,-0.526439,0.549161,-0.948939
294,-1.130523,1.302683,-1.838768,-0.008140,-0.204516,-1.507685,-0.639291,2.792122
231,0.681856,0.405061,0.642740,1.003382,2.617853,1.998825,-0.686829,1.139095


MODEL 1 : LOGISTIC REGRESSION

In [9]:
from sklearn.linear_model import LogisticRegression
y_pred = LogisticRegression(max_iter=10000).fit(X_train_std,y_train).predict(X_test_std)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def metrics_row(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
    }
print(metrics_row("Logistic Regression (scikit-learn)", y_test, y_pred))

{'Model': 'Logistic Regression (scratch)', 'Accuracy': 0.7532467532467533, 'Precision': 0.6666666666666666, 'Recall': 0.6181818181818182, 'F1': 0.6415094339622641}


MODEL 2 : KNN

In [15]:
from sklearn.neighbors import KNeighborsClassifier
y_pred=KNeighborsClassifier(n_neighbors=4).fit(X_train_std,y_train).predict(X_test_std)
print(metrics_row("KNN (scikit-learn) ", y_test, y_pred))

{'Model': 'KNN (scikit-learn) ', 'Accuracy': 0.7337662337662337, 'Precision': 0.6521739130434783, 'Recall': 0.5454545454545454, 'F1': 0.594059405940594}


MODEL : NAIVE BAYES


In [16]:
from sklearn.naive_bayes import GaussianNB
y_pred=GaussianNB().fit(X_train_std,y_train).predict(X_test_std)
print(metrics_row("Naive Bayes (scikit-learn)" , y_test , y_pred))

{'Model': 'Naive Bayes (scikit-learn)', 'Accuracy': 0.7597402597402597, 'Precision': 0.6551724137931034, 'Recall': 0.6909090909090909, 'F1': 0.672566371681416}
